In [23]:
import sys
import pandas as pd
import numpy as np
import torch

from transformers import (
    EarlyStoppingCallback,
    PatchTSTConfig,
    PatchTSTForPrediction,
    Trainer,
    TrainingArguments    
)

import optuna
from optuna.samplers import TPESampler

# Adiciona o diretório src ao path de forma mais robusta
sys.path.insert(0, '../')

from src import RepositorioDados
from src import evaluate_and_visualize, compute_metrics

# Detecta o dispositivo e a precisão usada nas operações
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.bfloat16 if (device.type == "cuda" and torch.cuda.is_bf16_supported()) else torch.float32
print(f"Device: {device} | Precision: {dtype}")

# Seed para resultados reproduzíveis
RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

Device: cpu | Precision: torch.float32


In [24]:
# 1. Features
LAGS = [3, 5, 7, 9]  # Lags de 1 a 5
USE_MEAN_FEATURES = [True, False]

# 2. Janelas temporais
CONTEXT_LENGTHS = [256, 275, 300, 325, 350]
FORECAST_HORIZONS = [1]  # Manter fixo por enquanto

# 3. Hiperparâmetros do modelo
MODEL_PARAMS = {
    'd_model':  [64, 128],
    'num_attention_heads': [16],
    'num_hidden_layers': [2, 5],
    'ffn_dim': [128, 256],
    'dropout': [0.05, 0.1],
    'patch_length': [8],
}

# 4. Hiperparâmetros de treinamento
TRAINING_PARAMS = {
    'learning_rate': [5e-4],
    'batch_size': [32, 64],
}

# Configurações fixas
FIXED_PARAMS = {
    'timestamp_col': 'timestamp',
    'target_col': ['Vol'],
    'id_columns': [],
    'forecast_horizon': 1,
    'epochs': 10,  # Reduzido para grid search
    'early_stopping_patience': 5,
    'num_workers': 0,
    'train_frac': 0.7,
    'valid_frac': 0.1,
}

print("Espaços de busca definidos")
print(f"Total de combinações de features: {len(LAGS) * len(USE_MEAN_FEATURES)}")
print(f"Total de combinações de context_length: {len(CONTEXT_LENGTHS)}")
print(f"Total de combinações de modelo: {np.prod([len(v) for v in MODEL_PARAMS.values()])}")
print(f"Total de combinações de treinamento: {np.prod([len(v) for v in TRAINING_PARAMS.values()])}")
total = len(LAGS) * len(USE_MEAN_FEATURES) * len(CONTEXT_LENGTHS) * np.prod([len(v) for v in MODEL_PARAMS.values()]) * np.prod([len(v) for v in TRAINING_PARAMS.values()])
print(f"\n🔍 Total de experimentos: {int(total)}")

Espaços de busca definidos
Total de combinações de features: 8
Total de combinações de context_length: 5
Total de combinações de modelo: 16
Total de combinações de treinamento: 2

🔍 Total de experimentos: 1280


In [25]:
import pandas as pd

pd.set_option('display.max_columns', None)

results_opt = pd.read_csv("./optuna_results_v2.csv")

results_opt.sort_values('eval_RMSE').head()

,status,error,experiment_id,lags,num_features,use_mean_features,context_length,forecast_horizon,d_model,num_attention_heads,num_hidden_layers,ffn_dim,dropout,patch_length,learning_rate,batch_size,train_loss,eval_loss,eval_MSE,eval_MAE,eval_RMSE,eval_MAPE,epochs_trained
17,success,NaN,17.0,3.0,3.0,False,300.0,1.0,128.0,16.0,2.0,256.0,0.05,8.0,0.0005,32.0,0.583428,0.024682,0.024682,0.087462,0.157106,47.291943,0.0
15,success,NaN,15.0,3.0,3.0,False,300.0,1.0,128.0,16.0,2.0,256.0,0.05,8.0,0.0005,32.0,0.583428,0.024682,0.024682,0.087462,0.157106,47.291943,0.0
14,success,NaN,14.0,3.0,3.0,False,300.0,1.0,128.0,16.0,2.0,256.0,0.05,8.0,0.0005,32.0,0.583428,0.024682,0.024682,0.087462,0.157106,47.291943,0.0
33,success,NaN,33.0,3.0,3.0,False,300.0,1.0,128.0,16.0,2.0,256.0,0.05,8.0,0.0005,32.0,0.583428,0.024682,0.024682,0.087462,0.157106,47.291943,0.0
13,success,NaN,13.0,3.0,3.0,False,300.0,1.0,128.0,16.0,2.0,128.0,0.10,8.0,0.0005,32.0,0.587165,0.025103,0.025103,0.088295,0.158439,48.621649,0.0


In [26]:
def run_experiment(
    repo,
    context_length,
    forecast_horizon,
    d_model,
    num_attention_heads,
    num_hidden_layers,
    ffn_dim,
    dropout,
    patch_length,
    learning_rate,
    batch_size,
    experiment_id,
    use_mean_features,
    lags
    ,config_params=FIXED_PARAMS,
):
    """
    Executa um experimento completo com os parâmetros fornecidos.
    Retorna um dicionário com os resultados.
    """
    lags_count = int(lags) if isinstance(lags, int) else len(lags)
    
    print(f"\n{'='*80}")
    print(f"🧪 EXPERIMENTO {experiment_id}")
    print(f"{'='*80}")
    print(f"Context Length: {context_length} | Horizon: {forecast_horizon}")
    print(f"d_model: {d_model}, heads: {num_attention_heads}, layers: {num_hidden_layers}")
    print(f"LR: {learning_rate}, Batch: {batch_size}, Patch: {patch_length}")
    print(f"{'='*80}\n")
    
    try:
        # 1. Preparar dados
        tsp, train_ds, valid_ds, test_ds, _, _, _ = repo.executar(
            timestamp_col=config_params['timestamp_col'],
            train_frac=config_params['train_frac'],
            valid_frac=config_params['valid_frac'],
            context_length=context_length,
            target_col=config_params['target_col'],
            id_columns=config_params['id_columns'],
            forecast_horizon=forecast_horizon,
            use_mean_features=use_mean_features,
            lags=lags)
        
        # 2. Configurar modelo
        config = PatchTSTConfig(
            context_length=context_length,
            patch_length=patch_length,
            num_input_channels=len(config_params['target_col']),
            patch_stride=patch_length,
            prediction_length=forecast_horizon,
            d_model=d_model,
            num_attention_heads=num_attention_heads,
            num_hidden_layers=num_hidden_layers,
            ffn_dim=ffn_dim,
            dropout=dropout,
            head_dropout=dropout,
            pooling_type=None,
            channel_attention=True, # Ativação da atenção entre canais
            loss='mse',
            pre_norm=True,
            norm_type='batchnorm',
        )
        
        model = PatchTSTForPrediction(config=config).to(device).to(dtype)
        
        # 3. Configurar treinamento
        train_args = TrainingArguments(
            output_dir="./grid_search_temp",
            overwrite_output_dir=True,
            learning_rate=learning_rate,
            num_train_epochs=config_params['epochs'],
            do_eval=True,
            eval_strategy="epoch",
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            dataloader_num_workers=config_params['num_workers'],
            save_strategy="no",
            logging_strategy="epoch",
            logging_dir=None,
            load_best_model_at_end=False,
            metric_for_best_model="eval_loss",
            greater_is_better=False,
            label_names=["future_values"],
            report_to="none",
            fp16=(dtype == torch.float16),
            bf16=(dtype == torch.bfloat16),
        )
        
        early_stopping = EarlyStoppingCallback(
            early_stopping_patience=config_params['early_stopping_patience'],
            early_stopping_threshold=0.001
        )
        
        trainer = Trainer(
            model=model,
            args=train_args,
            train_dataset=train_ds,
            eval_dataset=valid_ds,
            compute_metrics=compute_metrics,
            callbacks=[early_stopping]
        )
        
        # 4. Treinar
        train_result = trainer.train()
        
        # 5. Avaliar no conjunto de validação
        eval_result = trainer.evaluate()
        
        # 6. Coletar métricas
        result = {
            'experiment_id': experiment_id,
            'lags': lags,
            'num_features': lags_count + (1 if use_mean_features else 0),
            'use_mean_features': use_mean_features,
            'context_length': context_length,
            'forecast_horizon': forecast_horizon,
            'd_model': d_model,
            'num_attention_heads': num_attention_heads,
            'num_hidden_layers': num_hidden_layers,
            'ffn_dim': ffn_dim,
            'dropout': dropout,
            'patch_length': patch_length,
            'learning_rate': learning_rate,
            'batch_size': batch_size,
            'train_loss': train_result.training_loss,
            'eval_loss': eval_result['eval_loss'],
            'eval_MSE': eval_result.get('eval_MSE', None),
            'eval_MAE': eval_result.get('eval_MAE', None),
            'eval_RMSE': eval_result.get('eval_RMSE', None),
            'eval_MAPE': eval_result.get('eval_MAPE', None),
            'epochs_trained': train_result.global_step // len(train_ds) * batch_size,
            'status': 'success'
        }
        
        print(f"✅ Experimento {experiment_id} concluído!")
        print(f"   Val Loss: {eval_result['eval_loss']:.6f} | RMSE: {result['eval_RMSE']:.6f}")
        
        # Limpar memória
        del model, trainer, train_ds, valid_ds, test_ds, tsp
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
        
        return result
        
    except Exception as e:
        print(f"❌ Erro no experimento {experiment_id}: {str(e)}")
        return {
            'experiment_id': experiment_id,
            'lags': lags,
            'num_features': lags_count + (1 if use_mean_features else 0),
            'use_mean_features': use_mean_features,
            'context_length': context_length,
            'forecast_horizon': forecast_horizon,
            'd_model': d_model,
            'learning_rate': learning_rate,
            'batch_size': batch_size,
            'status': 'failed',
            'error': str(e)
        }

print("✓ Função run_experiment definida (com modo rápido)")

✓ Função run_experiment definida (com modo rápido)


In [27]:
def optuna_objective(trial):
    """Função objetivo para Optuna. Retorna a métrica a ser MINIMIZADA."""

    lags = trial.suggest_categorical('lags', LAGS)
    use_mean_features = trial.suggest_categorical('use_mean_features', USE_MEAN_FEATURES)

    context_length = trial.suggest_categorical('context_length', CONTEXT_LENGTHS)
    forecast_horizon = trial.suggest_categorical('forecast_horizon', FORECAST_HORIZONS)

    d_model = trial.suggest_categorical('d_model', MODEL_PARAMS['d_model'])
    num_attention_heads = trial.suggest_categorical('num_attention_heads', MODEL_PARAMS['num_attention_heads'])
    num_hidden_layers = trial.suggest_categorical('num_hidden_layers', MODEL_PARAMS['num_hidden_layers'])
    ffn_dim = trial.suggest_categorical('ffn_dim', MODEL_PARAMS['ffn_dim'])
    dropout = trial.suggest_categorical('dropout', MODEL_PARAMS['dropout'])
    patch_length = trial.suggest_categorical('patch_length', MODEL_PARAMS['patch_length'])

    learning_rate = trial.suggest_categorical('learning_rate', TRAINING_PARAMS['learning_rate'])
    batch_size = trial.suggest_categorical('batch_size', TRAINING_PARAMS['batch_size'])

    # Filtrar combinações inválidas (heads devem dividir d_model)
    if d_model % num_attention_heads != 0:
        trial.set_user_attr('full_result', {'status': 'failed', 'error': 'heads_not_divisible'})
        return float('inf')

    # Executar experimento
    result = run_experiment(
        repo=RepositorioDados(),
        context_length=context_length,
        forecast_horizon=forecast_horizon,
        d_model=d_model,
        num_attention_heads=num_attention_heads,
        num_hidden_layers=num_hidden_layers,
        ffn_dim=ffn_dim,
        dropout=dropout,
        patch_length=patch_length,
        learning_rate=learning_rate,
        batch_size=batch_size,
        experiment_id=trial.number,
        use_mean_features=use_mean_features,
        lags=lags
    )

    # Salvar resultado completo
    trial.set_user_attr('full_result', result)

    if result['status'] == 'failed':
        return float('inf')

    return result['eval_RMSE']  # ou 'eval_loss'

In [28]:
study = optuna.create_study(
    direction='minimize',  # Minimizar RMSE
    sampler=TPESampler(seed=RANDOM_STATE),
    study_name='patchtst_optimization'
)

[I 2026-02-17 11:28:56,830] A new study created in memory with name: patchtst_optimization


In [29]:
from datetime import datetime
start_time = datetime.now()

# Executar otimização
study.optimize(
    optuna_objective,
    n_trials=100,  # ⚡ REDUZIDO: 200 → 50
    show_progress_bar=True,
    callbacks=[
        lambda study, trial: study.trials_dataframe().to_csv(
            'optuna_results_partial.csv', index=False
        ) if trial.number % 5 == 0 else None
    ],
    gc_after_trial=True  # ⚡ NOVO: Libera memória após cada trial
)

end_time = datetime.now()
duration = end_time - start_time

print(f"\n{'='*80}")
print(f"✅ Busca Optuna concluída!")
print(f"⏱️ Tempo total: {duration}")
print(f"🏆 Melhor RMSE: {study.best_value:.6f}")
print(f"{'='*80}\n")

# Extrair resultados
results = [trial.user_attrs.get('full_result') for trial in study.trials 
            if 'full_result' in trial.user_attrs]

# Salvar resultados
df_results = pd.DataFrame(results)
df_results.to_csv('optuna_results_v3.csv', index=False)

# Mostrar melhores parâmetros
print("🏆 MELHORES HIPERPARÂMETROS (Optuna):")
print("="*80)
for key, value in study.best_params.items():
    print(f"{key:25s}: {value}")
print("="*80)

# Visualização Optuna
try:
    from optuna.visualization import plot_optimization_history, plot_param_importances
    
    # Histórico de otimização
    fig1 = plot_optimization_history(study)
    fig1.write_image('optuna_history.png')
    fig1.show()
    
    # Importância dos parâmetros
    fig2 = plot_param_importances(study)
    fig2.write_image('optuna_importance.png')
    fig2.show()
    
    print("📊 Gráficos salvos: optuna_history.png, optuna_importance.png")
except Exception as e:
    print(f"⚠️ Não foi possível gerar visualizações Optuna: {e}")
    print("   Instale: pip install plotly kaleido")

  0%|          | 0/100 [00:00<?, ?it/s]


🧪 EXPERIMENTO 0
Context Length: 275 | Horizon: 1
d_model: 64, heads: 16, layers: 2
LR: 0.0005, Batch: 64, Patch: 8

Carregando dados de C:\Users\Lenovo\Documents\GitHub\iniciacao-cientifica\data\raw\BTCUSDT_5m.txt
Calculando features de volatilidade


Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.


Epoch,Training Loss,Validation Loss,Mse,Mae,Rmse,Mape
1,0.798400,0.037508,0.037508,0.109009,0.193671,65.620553
2,0.575900,0.031902,0.031902,0.095899,0.178610,60.501075
3,0.555100,0.028690,0.028690,0.085706,0.169380,49.023366
4,0.700700,0.027754,0.027754,0.085786,0.166595,46.603903
5,0.542100,0.026788,0.026788,0.081293,0.163671,46.905085
6,0.525800,0.026078,0.026078,0.079853,0.161486,47.166654
7,0.532800,0.026426,0.026426,0.089637,0.162561,45.388156
8,0.523600,0.026189,0.026189,0.082025,0.161829,47.840881


  0%|          | 0/100 [01:05<?, ?it/s]

...

Epoch,Training Loss,Validation Loss,Mse,Mae,Rmse,Mape
1,0.676100,0.035123,0.035123,0.098877,0.187410,57.281280
2,0.594900,0.033148,0.033148,0.102221,0.182066,53.229994
3,0.598800,0.028785,0.028785,0.079226,0.169661,47.900838
4,0.558500,0.028832,0.028832,0.089060,0.169799,49.012685
5,0.566300,0.028771,0.028771,0.099938,0.169619,45.445007
6,0.544200,0.028598,0.028598,0.084020,0.169109,49.053100
7,0.549200,0.028979,0.028979,0.094035,0.170232,45.627034


Best trial: 68. Best value: 0.147071:  91%|█████████ | 91/100 [1:45:01<18:22, 122.48s/it]

[W 2026-02-17 13:13:57,840] Trial 91 failed with parameters: {'lags': 7, 'use_mean_features': True, 'context_length': 256, 'forecast_horizon': 1, 'd_model': 128, 'num_attention_heads': 16, 'num_hidden_layers': 2, 'ffn_dim': 128, 'dropout': 0.1, 'patch_length': 8, 'learning_rate': 0.0005, 'batch_size': 32} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\Lenovo\Documents\GitHub\iniciacao-cientifica\venv\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\Lenovo\AppData\Local\Temp\ipykernel_18780\822466487.py", line 26, in optuna_objective
    result = run_experiment(
             ^^^^^^^^^^^^^^^
  File "C:\Users\Lenovo\AppData\Local\Temp\ipykernel_18780\1616315580.py", line 105, in run_experiment
    train_result = trainer.train()
                   ^^^^^^^^^^^^^^^
  File "c:\Users\Lenovo\Documents\GitHub\iniciacao-cientifica\venv

Best trial: 68. Best value: 0.147071:  91%|█████████ | 91/100 [1:45:05<10:23, 69.29s/it] 


KeyboardInterrupt: 